# Recurrent Energy NodeField ablation

Thin driver for the repository implementation. Default execution runs only the seed-0 smoke matrix (100 graphs, ten epochs). Set `RUN_FULL = True` to train five seeds on 1,000 graphs and run the complete grid. Every execution creates a new result directory.

**Predefined outcome:** feasible structural condition match over all attempts. A positive paired seed-level 95% interval and an absolute effect of at least 0.05 are the full-run evidence criterion. The single-seed smoke run cannot establish significance. State changes measure empirical stabilization, not mathematical convergence.


## 0 — Experiment metadata

In [ ]:
EXPERIMENT_NAME = "recurrent_energy_nodefield_ablation_v1"
SEEDS = [0, 1, 2, 3, 4]
RUN_FULL = False


## 1 — Imports and reproducibility
Repository helpers seed Python, NumPy, Torch and CUDA. Unsupported deterministic operations emit warnings; environment and resolved configurations are saved with each run.

In [ ]:
from conditional_node_field_graph_generator.extensions.demo.recurrent_experiments import (
    RecurrentExperiment, summarize_results, plot_results, load_results,
    analysis_section, decision_report,
)
experiment = RecurrentExperiment(smoke=not RUN_FULL)
print(experiment.run_dir)
K_TRAIN = experiment.configs["recurrent_energy_annealed"]["model"]["recurrent_training_steps"]


## 2 — One canonical dataset
Use the existing cycle/path/star generator and fitted vectorizers. The cached 80/10/10 split, supervision and training-only preprocessing are shared by every model.

In [ ]:
experiment.prepare()
print({name: len(indices) for name, indices in experiment.splits.items()})


## 3 — Primary model matrix

| ID | Model | Memory | Corruption | Energy |
|---|---|---|---|---|
| A | Baseline | No | Existing fixed sigma | Yes |
| B | RENF | Yes | Constant | Yes |
| C | RENF | Yes | Annealed | Yes |
| D | Same checkpoint as C | Intervention-dependent | Annealed | Yes |

D is an alias, not another training run. Parameter counts are measured after setup. Raw RENF has extra parameters; the notebook does not claim parameter matching.


In [ ]:
print({name: config["model"] for name, config in experiment.configs.items()})


## 4 — Sanity checks before training
Abort on nonfinite loss or gradients. Check shapes, padding, hidden influence and score gradients. Parameter-sharing and finite-difference checks are covered by the prerequisite test suite.

In [ ]:
experiment.sanity_checks()


## 5 — Train and retain checkpoints
Smoke: equal ten-epoch budgets. Full: identical validation selection and patience; all epoch checkpoints are kept for matched-update curriculum comparisons. No test data selects checkpoints.

In [ ]:
experiment.train()


## 6 — Fixed-depth comparison and recorded evaluation
Runs the selected smoke/full matrix. Failed decodes remain rows. Unaligned generated graphs use label distributions rather than node-wise label accuracy.

In [ ]:
experiment.evaluate()
summary = summarize_results(experiment.run_dir)
results, diagnostics = load_results(experiment.run_dir)
analysis_section(results, diagnostics, "fixed_depth", k_train=K_TRAIN)


## 7 — Inference-depth scaling
The same checkpoints are evaluated beyond training depth. Full depths extend to 256; stop on nonfinite states.

In [ ]:
analysis_section(results, diagnostics, "depth", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 8 — Hidden-state reset
Reset occurs before the zero-based designated evaluation. Compare head count errors and saved per-step quality around the intervention.

In [ ]:
analysis_section(results, diagnostics, "reset", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 9 — Hidden-state shuffle
Full mode runs three independent within-graph permutations at each reset fraction. Smoke mode leaves this analysis empty.

In [ ]:
analysis_section(results, diagnostics, "shuffle", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 10 — Observable-state destruction
Full mode includes the complete persistent/fresh-x × persistent/reset-h grid. The smoke matrix contains normal, midpoint resets, and fresh-x every step.

In [ ]:
analysis_section(results, diagnostics, "channels", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 11 — Training curriculum
Compare constant and annealed models under normal inference, fresh-x noise, and increased depth. Full mode also compares retained checkpoints at matched updates.

In [ ]:
analysis_section(results, diagnostics, "curriculum", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 12 — Memory versus repeated computation
Full mode resets h before every evaluation on the same trained checkpoint.

In [ ]:
analysis_section(results, diagnostics, "no_memory", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 13 — Inference noise
Fresh x is N(0, s²I), with s recorded explicitly. Replacement noise and Langevin noise use distinct controls. All inference x values use the model’s scaled feature space.

In [ ]:
analysis_section(results, diagnostics, "noise", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 14 — State stability
Inspect hidden delta, score norm, potential, and prediction changes. These are empirical diagnostics; potential need not decrease when memory changes.

In [ ]:
analysis_section(results, diagnostics, "stability", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 15 — Anytime computation
Full mode selects stopping thresholds using validation trajectories, then measures test steps saved and mean/worst quality loss. Decoder-unchanged stopping uses three consecutive unchanged solutions. This secondary experiment is not executed in smoke mode.

In [ ]:
analysis_section(results, diagnostics, "anytime", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 16 — Statistical tables
Full comparisons aggregate independent training seeds and use paired seed-level confidence intervals. A one-seed run has undefined across-seed standard deviations and confidence intervals.

In [ ]:
analysis_section(results, diagnostics, "statistics", run_dir=experiment.run_dir, k_train=K_TRAIN)


## 17 — Figures
All values come from saved results, diagnostics and training history. Re-run this cell to regenerate the figures without retraining.

In [ ]:
plot_results(experiment.run_dir)


## 18 — Decision criteria
The predefined effect criterion is applied only to full, independent-seed comparisons. Memory intervention evidence remains an intervention-based interpretation. Preservation under fresh-x corruption is an open experimental question, not an expected result.

In [ ]:
decision_report(experiment.run_dir)
